# Problème de  Helmolz
$$\Delta u + k²u = 0 \hspace{2mm} sur \hspace{2mm} \Omega$$
$$\frac{\partial u}{\partial n} = 0 \hspace{2mm} sur \hspace{2mm} \Gamma_N$$
$$\frac{\partial u}{\partial n} + iku = 0 \hspace{2mm} sur \hspace{2mm} \Gamma_D$$
$$u = 1 \hspace{2mm} sur \hspace{2mm} \Gamma_G \hspace{2mm}$$
où on définit les ensembles $\Omega = [0, 1]², \Gamma_N = 0 \times [0, 1] \bigsqcup 1 \times [0, 1]$ (u nulle en haut et en bas), $\Gamma_D = [0, 1] \times 1$ (droite), et enfin $\Gamma_G = [0, 1] \times 0$ (gauche)

In [ ]:
from pathlib import Path

from mpi4py import MPI
from petsc4py.PETSc import ScalarType  # type: ignore

import numpy as np

import ufl
from dolfinx import fem, io, mesh, plot
from dolfinx.fem.petsc import LinearProblem

msh = mesh.create_rectangle(
    comm=MPI.COMM_WORLD,
    points=((0.0, 0.0), (1.0, 1.0)),
    n=(32, 16),
    cell_type=mesh.CellType.triangle,
)
V = fem.functionspace(msh, ("Lagrange", 1))

In [ ]:


mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, 0, gdim=3)
domain = mesh_data.mesh
assert mesh_data.facet_tags is not None
facet_tags = mesh_data.facet_tags

In [ ]:
V = fem.functionspace(domain, ("Lagrange", 1))

# Discrete frequency range
freq = np.arange(10, 1000, 5)  # Hz

# Air parameters
# rho0 = 1.225  # kg/m^3
# c = 340  # m/s

In [ ]:
# Impedance calculation
def delany_bazley_layer(f, rho0, c, sigma):
    X = rho0 * f / sigma
    # Zc = rho0 * c * (1 + 0.0571 * X**-0.754 - 1j * 0.087 * X**-0.732)
    # kc = 2 * np.pi * f / c * (1 + 0.0978 * (X**-0.700) - 1j * 0.189 * (X**-0.595))
    kc = 3
    # Z_s = -1j * Zc * (1 / np.tan(kc * d))
    return Z_s


sigma = 1.5e4
d = 0.01
Z_s = delany_bazley_layer(freq, rho0, c, sigma)

In [ ]:
omega = fem.Constant(domain, default_scalar_type(0))
k = fem.Constant(domain, default_scalar_type(0))
Z = fem.Constant(domain, default_scalar_type(0))
v_n = 1e-5

In [ ]:
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)

In [ ]:
p = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

a = (
    ufl.inner(ufl.grad(p), ufl.grad(v)) * ufl.dx
    # + 1 * rho0 * omega / Z * ufl.inner(p, v) * ds(Z_bc_tag)
    - k**2 * ufl.inner(p, v) * ufl.dx
    + 1j*k*ufl.inner(p, v)*ds("right_marker")
)
L = -1 * omega * rho0 * ufl.inner(v_n, v) * ds(v_bc_tag)

In [ ]:
p_a = fem.Function(V)
p_a.name = "pressure"

problem = LinearProblem(
    a,
    L,
    u=p_a,
    petsc_options={
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
    },
    petsc_options_prefix="helmholtz",
)